In [115]:
import os
import json
import numpy as np
import torch
import pandas as pd
import cv2
import trimesh
from pathlib import Path
from PIL import Image
from scipy.spatial.transform import Rotation

In [116]:
PROJECT_ROOT = Path(".").resolve().parent

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"

MESH_PATH = PROJECT_ROOT / "megapose_objects/lamp/mesh.ply"
CAM_INFO  = PROJECT_ROOT / "outputs/camera_info.json"

DETECTION_PROMPT     = "yellow lamp"
BOX_THRESHOLD        = 0.30
TEXT_THRESHOLD       = 0.25
N_REFINER_ITERATIONS = 5
DEVICE               = "cuda"

# Known target pose from the .g file (world frame): [x, y, z, qw, qx, qy, qz]
TARGET_POSE_RAI = np.array([0.219547, 0.280159, 0.698313, 0.38692, 0.0, 0.0, -0.922113])

In [117]:
def rai_pose_to_matrix(pose: np.ndarray) -> np.ndarray:
    pos = pose[:3]
    qw, qx, qy, qz = pose[3], pose[4], pose[5], pose[6]
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = pos
    return T


def boxes_cxcywh_to_xyxy(boxes_norm: torch.Tensor, width: int, height: int) -> torch.Tensor:
    """Convert GroundingDINO [cx,cy,w,h] normalised -> pixel [x1,y1,x2,y2]."""
    cx, cy, w, h = boxes_norm.unbind(-1)
    x1 = (cx - w / 2) * width
    y1 = (cy - h / 2) * height
    x2 = (cx + w / 2) * width
    y2 = (cy + h / 2) * height
    return torch.stack([x1, y1, x2, y2], dim=-1)


def project_pt(pt_3d, K):
    p = K @ np.array(pt_3d)
    return (int(p[0] / p[2]), int(p[1] / p[2]))

### 1. Load GroundingDINO

In [118]:
from groundingdino.util.inference import load_model, load_image, predict

print("Loading GroundingDINO …")
gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
gd_model.eval()
print("Done.")

Loading GroundingDINO …
final text_encoder_type: bert-base-uncased


Done.


### 2. Detect Lamp Across All Cameras

In [119]:
with open(CAM_INFO) as f:
    camera_info = json.load(f)

print(f"Running GroundingDINO with prompt: '{DETECTION_PROMPT}'")

best = {"score": -1.0, "cam": None, "box_xyxy": None}

for cam_name, info in camera_info.items():
    rgb_path = info["rgb_path"]
    W, H     = info["width"], info["height"]

    image_source, image_tensor = load_image(rgb_path)

    boxes_norm, logits, phrases = predict(
        model=gd_model,
        image=image_tensor,
        caption=DETECTION_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
    )

    if len(logits) == 0:
        print(f"  {cam_name}: no detection")
        continue

    best_idx   = logits.argmax().item()
    best_score = logits[best_idx].item()
    best_box   = boxes_cxcywh_to_xyxy(boxes_norm[best_idx].unsqueeze(0), W, H)[0]

    print(f"  {cam_name}: score={best_score:.3f}  box={best_box.numpy().astype(int).tolist()}")

    if best_score > best["score"]:
        best.update({"score": best_score, "cam": cam_name, "box_xyxy": best_box})

if best["cam"] is None:
    raise RuntimeError(
        "GroundingDINO found no detections. "
        "Lower BOX_THRESHOLD or change the prompt."
    )

print(f"\nBest detection: {best['cam']}  score={best['score']:.3f}")

Running GroundingDINO with prompt: 'yellow lamp'


  cam_dim_0: score=0.539  box=[255, 1280, 331, 1391]
  cam_dim_1: score=0.510  box=[96, 405, 1824, 769]
  cam_dim_2: score=0.532  box=[97, 423, 1823, 767]
  cam_dim_3: score=0.455  box=[96, 400, 1824, 768]
  cam_dim_4: score=0.452  box=[96, 425, 1823, 767]

Best detection: cam_dim_0  score=0.539


### 3. Load MegaPose

In [120]:
# Jupyter kernels don't inherit ~/.bashrc — set HAPPYPOSE_DATA_DIR explicitly.
# Run `echo $HAPPYPOSE_DATA_DIR` in a terminal on the server to get the value.
if "HAPPYPOSE_DATA_DIR" not in os.environ:
    os.environ["HAPPYPOSE_DATA_DIR"] = "/home/salman/rai_workspace/happypose_data"  # <-- update if different

print("HAPPYPOSE_DATA_DIR:", os.environ["HAPPYPOSE_DATA_DIR"])

HAPPYPOSE_DATA_DIR: /home/salman/rai_workspace/happypose_data


In [121]:
from happypose.toolbox.datasets.object_dataset import RigidObjectDataset, RigidObject
from happypose.toolbox.utils.load_model import load_named_model
from happypose.toolbox.inference.types import ObservationTensor
from happypose.toolbox.utils.tensor_collection import PandasTensorCollection

print("Loading MegaPose …")

object_dataset = RigidObjectDataset([
    RigidObject(
        label="lamp",
        mesh_path=MESH_PATH,
        scaling_factor=1.0,
    )
])

# megapose-1.0-RGBD uses the depth refiner — better accuracy than RGB-only.
pose_estimator = load_named_model(
    "megapose-1.0-RGBD",
    object_dataset=object_dataset,
)
# load_named_model moves sub-models to CUDA but not the PoseEstimator wrapper.
# _SO3_grid is a registered buffer on the wrapper — move it manually.
pose_estimator._SO3_grid = pose_estimator._SO3_grid.to(DEVICE)
print("Done.")

Loading MegaPose …


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
Xlib:  extension "XFree86-DGA" missing on display ":0.0".
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Known pipe types:
  glxGraphicsPipe
(1 aux display modules not yet loaded.)
Xlib:  extension "XFree86-DGA" missing on display ":0.0".
huggingface/tokenizers: The current process just got forked, after parallelism has a

Done.


### 4. Prepare Observation & Run MegaPose

In [122]:
cam_name = best["cam"]
info     = camera_info[cam_name]

rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
depth = np.load(info["depth_path"])
K     = np.array(info["K"], dtype=np.float64)

observation = ObservationTensor.from_numpy(
    rgb=rgb,
    depth=depth,
    K=K,
).to(DEVICE)

box_xyxy = best["box_xyxy"].unsqueeze(0).float().to(DEVICE)

detections = PandasTensorCollection(
    infos=pd.DataFrame({
        "label":       ["lamp"],
        "batch_im_id": [0],
        "instance_id": [0],
        "score":       [float(best["score"])],
    }),
    bboxes=box_xyxy,
)

In [123]:
print("Running MegaPose coarse + refiner …")
output, _ = pose_estimator.run_inference_pipeline(
    observation,
    detections=detections,
    run_detector=False,
    n_refiner_iterations=N_REFINER_ITERATIONS,
    n_pose_hypotheses=1,
)

T_cam_obj = output.poses[0].cpu().numpy()
print("Done.")

Running MegaPose coarse + refiner …
Done.


### 5. Convert to World Frame & Evaluate

In [124]:
T_world_cam = np.array(info["T_world_cam"])
T_world_obj = T_world_cam @ T_cam_obj

position  = T_world_obj[:3, 3]
R_mat     = T_world_obj[:3, :3]
quat_xyzw = Rotation.from_matrix(R_mat).as_quat()        # [x, y, z, w]
quat_wxyz = np.array([quat_xyzw[3], *quat_xyzw[:3]])     # RAI convention

print("=" * 60)
print("ESTIMATED POSE (world frame)")
print("=" * 60)
print(f"Position    [x, y, z] : {position}")
print(f"Quaternion  [w,x,y,z] : {quat_wxyz}")
print(f"\n4x4 matrix:\n{T_world_obj}")

# Compare against ground-truth position from .g file
TRUE_OBJ_POSE_RAI = np.array([-0.620966, -0.342511, 0.671156,
                               0.287543, -0.46152, 0.714029, 0.441])
true_pos = TRUE_OBJ_POSE_RAI[:3]
pos_err  = np.linalg.norm(position - true_pos)

print(f"\nTrue object position (from .g): {true_pos}")
print(f"Estimated position:             {np.round(position, 4)}")
print(f"Position error:                 {pos_err*100:.1f} cm")
print(f"\nTarget position (goal pose):    {TARGET_POSE_RAI[:3]}")

ESTIMATED POSE (world frame)
Position    [x, y, z] : [-0.61998516 -0.34027916  0.68286133]
Quaternion  [w,x,y,z] : [ 0.31881487 -0.49191166  0.67919928  0.44166541]

4x4 matrix:
[[-0.31276    -0.94983113 -0.00144306 -0.61998516]
 [-0.38659304  0.12590916  0.91361517 -0.34027916]
 [-0.8675983   0.28630015 -0.4065775   0.68286133]
 [ 0.          0.          0.          1.        ]]

True object position (from .g): [-0.620966 -0.342511  0.671156]
Estimated position:             [-0.62   -0.3403  0.6829]
Position error:                 1.2 cm

Target position (goal pose):    [0.219547 0.280159 0.698313]


### 6. Visualization

In [125]:
print("Generating visualization …")

vis_img = cv2.imread(info["rgb_path"])   # BGR
H_vis, W_vis = vis_img.shape[:2]
K_vis = np.array(info["K"])

# Green detection bounding box
x1, y1, x2, y2 = best["box_xyxy"].cpu().numpy().astype(int)
cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 4)
cv2.putText(vis_img, f"GD score={best['score']:.2f}", (x1, max(y1 - 15, 30)),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

# XYZ axes at estimated pose (red=X, green=Y, blue=Z)
T        = T_cam_obj
axis_len = 0.05  # 5 cm
o_px = project_pt(T[:3, 3], K_vis)
x_px = project_pt(T[:3, 3] + axis_len * T[:3, 0], K_vis)
y_px = project_pt(T[:3, 3] + axis_len * T[:3, 1], K_vis)
z_px = project_pt(T[:3, 3] + axis_len * T[:3, 2], K_vis)
cv2.arrowedLine(vis_img, o_px, x_px, (0, 0, 255), 4, tipLength=0.3)   # X red
cv2.arrowedLine(vis_img, o_px, y_px, (0, 255, 0), 4, tipLength=0.3)   # Y green
cv2.arrowedLine(vis_img, o_px, z_px, (255, 0, 0), 4, tipLength=0.3)   # Z blue

# Project mesh vertices as cyan dots
mesh     = trimesh.load(str(MESH_PATH))
verts    = np.array(mesh.vertices)
verts_hom = np.hstack([verts, np.ones((len(verts), 1))])
verts_cam = (T_cam_obj @ verts_hom.T).T[:, :3]
front     = verts_cam[:, 2] > 0
if front.any():
    pts = (K_vis @ verts_cam[front].T).T
    pts = (pts[:, :2] / pts[:, 2:3]).astype(int)
    in_bounds = (pts[:, 0] >= 0) & (pts[:, 0] < W_vis) & \
                (pts[:, 1] >= 0) & (pts[:, 1] < H_vis)
    for pt in pts[in_bounds][::8]:   # every 8th vertex for speed
        cv2.circle(vis_img, tuple(pt), 3, (255, 255, 0), -1)

# Text overlay with estimated position
pos_str = f"pos=[{position[0]:.3f}, {position[1]:.3f}, {position[2]:.3f}]"
cv2.putText(vis_img, pos_str, (20, H_vis - 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
cv2.putText(vis_img, pos_str, (20, H_vis - 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

out_path = PROJECT_ROOT / "outputs" / "pose_visualization.png"
cv2.imwrite(str(out_path), vis_img)
print(f"Saved -> {out_path}")

Generating visualization …
Saved -> /home/salman/rai_workspace/Voxel_manipulation_v2/pose_estimation/outputs/pose_visualization.png


In [126]:
def depth_to_world_points(depth, K, T_world_cam, stride=4, max_depth=3.0):
    H, W = depth.shape

    ys, xs = np.meshgrid(
        np.arange(0, H, stride),
        np.arange(0, W, stride),
        indexing="ij"
    )

    zs = depth[ys, xs]

    valid = (zs > 0) & (zs < max_depth)

    xs = xs[valid]
    ys = ys[valid]
    zs = zs[valid]

    fx = K[0, 0]
    fy = K[1, 1]
    cx = K[0, 2]
    cy = K[1, 2]

    X = (xs - cx) * zs / fx
    Y = (ys - cy) * zs / fy
    Z = zs

    points_cam = np.stack([X, Y, Z], axis=1)

    points_cam_h = np.hstack([
        points_cam,
        np.ones((len(points_cam), 1))
    ])

    points_world = (T_world_cam @ points_cam_h.T).T[:, :3]

    return points_world

In [127]:
import numpy as np
import open3d as o3d


def align_normals_with_centroid(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)

    centroid = np.mean(pts, axis=0)
    for i in range(len(normals)):
        to_point = pts[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        _, idx, _ = kdtree.search_knn_vector_3d(pts[i], 10)
        avg_normal = np.mean(normals[idx], axis=0)
        norm = np.linalg.norm(avg_normal)

        if norm > 1e-12:
            avg_normal = avg_normal / norm
            if np.dot(normals[i], avg_normal) < 0:
                normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


In [ ]:

# 7. Convert estimated object mesh to point cloud for grasping

mesh = trimesh.load(str(MESH_PATH))

# sample object surface points
sampled_points, _ = trimesh.sample.sample_surface(mesh, 30000)

# transform object points from object frame to world frame
sampled_points_h = np.hstack([sampled_points, np.ones((len(sampled_points), 1))])
target_points_world = (T_world_obj @ sampled_points_h.T).T[:, :3]

target_pcd = o3d.geometry.PointCloud()
target_pcd.points = o3d.utility.Vector3dVector(target_points_world)

# downsample
pcd_tmp = target_pcd.voxel_down_sample(voxel_size=0.0025)

# estimate normals
pcd_tmp.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

pcd_tmp = align_normals_with_centroid(pcd_tmp)
pcd_tmp = reorient_normals_locally(pcd_tmp)

normals = np.asarray(pcd_tmp.normals)
cropped_pcl = np.asarray(pcd_tmp.points)


assert len(cropped_pcl) > 0, "Estimated target point cloud is empty"
#o3d.visualization.draw_geometries([pcd_tmp], window_name="Estimated Target Object with Normals", point_show_normal=True)
print("Target grasp point cloud:", cropped_pcl.shape)
print("Normals:", normals.shape)

[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] Failed to initialize GLEW.
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.
Target grasp point cloud: (2085, 3)
Normals: (2085, 3)


In [129]:
import matplotlib.pyplot as plt
def safe_norm(v, eps=1e-9):
    n = np.linalg.norm(v)
    if n < eps:
        return v
    return v / n


def angle_between(v1, v2):
    v1_u = safe_norm(v1)
    v2_u = safe_norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)


def compute_approach_clearance(center, approach_dir, all_points,
                               corridor_length=0.08,
                               corridor_radius=0.015,
                               ignore_radius=0.015):

    approach_dir = safe_norm(approach_dir)

    relative = all_points - center

    # candidate center'ın arkasındaki approach direction boyunca projection
    projection = relative @ approach_dir

    # grasp center'a çok yakın noktaları ignore ediyoruz,
    # çünkü onlar zaten grasp yapılacak objenin kendi noktaları
    valid_depth = (projection > ignore_radius) & (projection < corridor_length)

    perpendicular = relative - np.outer(projection, approach_dir)
    perpendicular_distance = np.linalg.norm(perpendicular, axis=1)

    inside_corridor = valid_depth & (perpendicular_distance < corridor_radius)

    blocking_count = np.sum(inside_corridor)

    # az blocking point iyidir, çok point kötüdür
    clearance_score = np.exp(-0.15 * blocking_count)

    return clearance_score, blocking_count
def compute_pca_axes(points):
    centered = points - np.mean(points, axis=0)

    covariance = np.cov(centered.T)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    # büyükten küçüğe sırala
    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    long_axis = eigenvectors[:, 0]
    mid_axis = eigenvectors[:, 1]
    short_axis = eigenvectors[:, 2]

    return long_axis, mid_axis, short_axis, eigenvalues

def score_candidate(p1, p2, n1, n2, centroid, target_points, obstacle_points, pca_axes,
                    min_distance=0.01,
                    max_distance=0.09,
                    ideal_width=0.05):

    center = (p1 + p2) / 2
    width = np.linalg.norm(p2 - p1)
    grasp_axis = safe_norm(p2 - p1)

    normal_angle = angle_between(n1, n2)
    angle_n1_axis = angle_between(n1, grasp_axis)
    angle_n2_axis = angle_between(n2, grasp_axis)

    reasons = []

    antipodal_score = np.clip((normal_angle - 160) / 20, 0, 1)

    projection_n1 = np.dot(n1, grasp_axis)
    projection_n2 = np.dot(n2, grasp_axis)
    alignment_score = np.clip(0.5 * ((-projection_n1) + projection_n2), 0, 1)

    width_error = abs(width - ideal_width)
    distance_score = np.clip(1 - width_error / (max_distance - min_distance), 0, 1)

    dist_to_centroid = np.linalg.norm(center - centroid)
    centroid_score = np.exp(-20 * dist_to_centroid)
    # Penalize grasps below the object centroid because they are likely to be blocked by table/floor.
    height_diff = center[2] - centroid[2]

    if height_diff >= 0:
        height_score = 1.0
    else:
        height_score = np.exp(60 * height_diff)
    long_axis, mid_axis, short_axis, eigenvalues = pca_axes

    # grasp axis objenin uzun eksenine değil, daha çok kısa/orta eksenine paralel olursa daha iyi
    align_long = abs(np.dot(grasp_axis, long_axis))
    align_mid = abs(np.dot(grasp_axis, mid_axis))
    align_short = abs(np.dot(grasp_axis, short_axis))

    pca_score = max(align_mid, align_short)
    approach_dir = safe_norm(-(n1 + n2))

    if np.linalg.norm(approach_dir) < 1e-6:
        approach_dir = np.array([0.0, 0.0, 1.0])

    self_clearance_score, self_blocking_count = compute_approach_clearance(
        center,
        approach_dir,
        target_points,
        corridor_length=0.08,
        corridor_radius=0.015,
        ignore_radius=0.015
    )

    scene_clearance_score, scene_blocking_count = compute_approach_clearance(
        center,
        approach_dir,
        obstacle_points,
        corridor_length=0.08,
        corridor_radius=0.015,
        ignore_radius=0.015
    )

    if antipodal_score > 0.8:
        reasons.append("good antipodal normals")
    else:
        reasons.append("weaker antipodal quality")

    if alignment_score > 0.8:
        reasons.append("good normal-axis alignment")
    else:
        reasons.append("weaker normal-axis alignment")

    if distance_score > 0.8:
        reasons.append("good grasp width")
    else:
        reasons.append("less ideal grasp width")

    if dist_to_centroid < 0.03:
        reasons.append("near object centroid")
    else:
        reasons.append("farther from object centroid")
    if height_score > 0.8:
        reasons.append("reachable height region")
    elif height_score > 0.4:
        reasons.append("slightly low grasp region")
    else:
        reasons.append("low grasp region penalized")
    if pca_score > 0.8:
        reasons.append("good alignment with object minor axis")
    elif align_long > 0.8:
        reasons.append("grasp axis aligned with long object axis")
    else:
        reasons.append("moderate PCA alignment")
    if self_blocking_count == 0:
        reasons.append("clear self approach")
    elif self_blocking_count < 10:
        reasons.append("partially blocked by target geometry")
    else:
        reasons.append("blocked by target geometry")

    if scene_blocking_count == 0:
        reasons.append("clear scene approach")
    elif scene_blocking_count < 10:
        reasons.append("partially blocked by scene clutter")
    else:
        reasons.append("blocked by scene clutter")

    score = (
        0.18 * antipodal_score +
        0.15 * alignment_score +
        0.10 * distance_score +
        0.05 * centroid_score +
        0.10 * pca_score +
        0.10 * self_clearance_score +
        0.17 * scene_clearance_score +
        0.15 * height_score
    )

    return {
        "score": score,
        "center": center,
        "width": width,
        "normal_angle": normal_angle,
        "angle_n1_axis": angle_n1_axis,
        "angle_n2_axis": angle_n2_axis,
        "dist_to_centroid": dist_to_centroid,
        "antipodal_score": antipodal_score,
        "alignment_score": alignment_score,
        "distance_score": distance_score,
        "centroid_score": centroid_score,
        "pca_score": pca_score,
        "align_long": align_long,
        "align_mid": align_mid,
        "align_short": align_short,
        "self_clearance_score": self_clearance_score,
        "self_blocking_count": self_blocking_count,
        "scene_clearance_score": scene_clearance_score,
        "scene_blocking_count": scene_blocking_count,
        "approach_dir": approach_dir,
        "reasons": reasons,
        "height_score": height_score,
        "height_diff": height_diff,
    }

def create_grasp_visualization(candidate, color=[0, 1, 0], radius=0.002):
    p1 = candidate[0]
    p2 = candidate[1]
    n1 = candidate[2]
    n2 = candidate[3]

    geometries = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere1.translate(p1)
    sphere1.paint_uniform_color(color)

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere2.translate(p2)
    sphere2.paint_uniform_color(color)

    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector([p1, p2])
    line.lines = o3d.utility.Vector2iVector([[0, 1]])
    line.colors = o3d.utility.Vector3dVector([color])

    normal_line1 = o3d.geometry.LineSet()
    normal_line1.points = o3d.utility.Vector3dVector([p1, p1 + 0.03 * n1])
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])
    normal_line1.colors = o3d.utility.Vector3dVector([color])

    normal_line2 = o3d.geometry.LineSet()
    normal_line2.points = o3d.utility.Vector3dVector([p2, p2 + 0.03 * n2])
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])
    normal_line2.colors = o3d.utility.Vector3dVector([color])

    geometries.extend([sphere1, sphere2, line, normal_line1, normal_line2])
    return geometries


min_distance = 0.01
max_distance = 0.09
small_angle_threshold = 10
large_angle_threshold = 160

top_k = 10

best_grasps = []
compute_grasp_points = True
cropped_pcl = np.asarray(pcd_tmp.points)

if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []
    scored_candidates = []

    centroid = np.mean(points, axis=0)

    target_points = points

    # Build scene point cloud from the depth image
    scene_points = depth_to_world_points(
        depth=depth,
        K=K,
        T_world_cam=T_world_cam,
        stride=4,
        max_depth=3.0
    )

    # Remove points near the target object so the object itself is not counted as scene clutter
    dist_to_target_centroid = np.linalg.norm(scene_points - centroid, axis=1)

    obstacle_points = scene_points[dist_to_target_centroid > 0.15]

    # Remove very low points below the object/table region if needed
    # Keep this conservative for now
    obstacle_points = obstacle_points[obstacle_points[:, 2] > centroid[2] - 0.12]

    # Downsample obstacle points for speed
    obstacle_pcd = o3d.geometry.PointCloud()
    obstacle_pcd.points = o3d.utility.Vector3dVector(obstacle_points)
    obstacle_pcd = obstacle_pcd.voxel_down_sample(voxel_size=0.02)
    obstacle_points = np.asarray(obstacle_pcd.points)

    pca_axes = compute_pca_axes(target_points)

    long_axis, mid_axis, short_axis, eigenvalues = pca_axes

    print("PCA eigenvalues:", eigenvalues)
    print("Long axis:", long_axis)
    print("Mid axis:", mid_axis)
    print("Short axis:", short_axis)

    print("Scene points from depth:", len(scene_points))
    print("Target points:", len(target_points))
    print("Obstacle points used for scene clearance:", len(obstacle_points))


    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)

            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)

                if large_angle_threshold <= angle <= 180:
                    epiline = safe_norm(p2 - p1)

                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                    if projection_n1 < 0 and projection_n2 > 0:
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)

                        if (
                            min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold
                            and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold
                        ):
                            info = score_candidate(
                                p1, p2, n1, n2, centroid, target_points, obstacle_points, pca_axes,
                                min_distance=min_distance,
                                max_distance=max_distance,
                                ideal_width=0.05
                            )

                            candidate = (
                                p1,
                                p2,
                                n1,
                                n2,
                                info["normal_angle"],
                                info["width"],
                                info["angle_n1_axis"],
                                info["angle_n2_axis"],
                                info["dist_to_centroid"],
                                info["score"],
                                info["reasons"],
                                info["self_clearance_score"],
                                info["self_blocking_count"],
                                info["scene_clearance_score"],
                                info["scene_blocking_count"],
                                info["pca_score"],
                                info["align_long"],
                                info["align_mid"],
                                info["align_short"]
                            )

                            grasp_candidates.append(candidate)
                            scored_candidates.append(candidate)

    if scored_candidates:
        scored_candidates = sorted(
            scored_candidates,
            key=lambda x: x[9],
            reverse=True
        )

        print("Number of valid grasp candidates:", len(scored_candidates))
        print("\nTop candidates:")

        for rank, cand in enumerate(scored_candidates[:top_k]):
            print("\nCandidate", rank)
            print("score:", round(cand[9], 4))
            print("width:", round(cand[5], 4))
            print("normal_angle:", round(cand[4], 2))
            print("dist_to_centroid:", round(cand[8], 4))
            print("self_clearance_score:", round(cand[11], 4))
            print("self_blocking_count:", cand[12])
            print("scene_clearance_score:", round(cand[13], 4))
            print("scene_blocking_count:", cand[14])
            print("pca_score:", round(cand[15], 4))
            print("align_long:", round(cand[16], 4))
            print("align_mid:", round(cand[17], 4))
            print("align_short:", round(cand[18], 4))
            print("reasons:", cand[10])


        blocked_candidates = [c for c in scored_candidates if c[12] > 0 or c[14] > 0]

        print("\nNumber of blocked candidates:", len(blocked_candidates))

        if blocked_candidates:
            print("\nExample blocked candidates:")

            for rank, cand in enumerate(blocked_candidates[:5]):
                print("\nBlocked Candidate", rank)
                print("score:", round(cand[9], 4))
                print("width:", round(cand[5], 4))
                print("normal_angle:", round(cand[4], 2))
                print("dist_to_centroid:", round(cand[8], 4))
                print("self_clearance_score:", round(cand[11], 4))
                print("self_blocking_count:", cand[12])
                print("scene_clearance_score:", round(cand[13], 4))
                print("scene_blocking_count:", cand[14])
                print("pca_score:", round(cand[15], 4))
                print("align_long:", round(cand[16], 4))
                print("align_mid:", round(cand[17], 4))
                print("align_short:", round(cand[18], 4))
                print("reasons:", cand[10])

        best_grasp = scored_candidates[0]
        best_grasps.append(best_grasp)
        # Visualize top-k grasp candidates
        top_k_vis = 5

        object_cloud = o3d.geometry.PointCloud()
        object_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
        object_cloud.paint_uniform_color([0.6, 0.6, 0.6])

        visual_geometries = [object_cloud]

        colors = [
            [0, 1, 0],      # best: green
            [0, 0, 1],      # blue
            [1, 0.7, 0],    # orange
            [1, 0, 1],      # purple
            [0, 1, 1]       # cyan
        ]

        for rank, cand in enumerate(scored_candidates[:top_k_vis]):
            visual_geometries.extend(
                create_grasp_visualization(
                    cand,
                    color=colors[rank % len(colors)],
                    radius=0.0025 if rank == 0 else 0.0018
                )
            )

        

        best = scored_candidates[0]
        p1 = best[0]
        p2 = best[1]

        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d')

        # object point cloud
        ax.scatter(
            cropped_pcl[:, 0],
            cropped_pcl[:, 1],
            cropped_pcl[:, 2],
            s=1,
            alpha=0.3
        )

        # grasp points
        ax.scatter(
            [p1[0], p2[0]],
            [p1[1], p2[1]],
            [p1[2], p2[2]],
            s=100
        )

        # line between grasp points
        ax.plot(
            [p1[0], p2[0]],
            [p1[1], p2[1]],
            [p1[2], p2[2]]
        )

        ax.set_title("Best Grasp Visualization")

        plt.savefig("best_grasp_visualization.png", dpi=300)
        plt.close()

        print("Saved best_grasp_visualization.png")

        print("\nSelected best grasp:")
        print(best_grasp)

    else:
        print("No suitable grasp candidates found")
        grasp_candidates.append(None)

PCA eigenvalues: [0.00084346 0.00028073 0.0002743 ]
Long axis: [-0.0649984   0.91412665 -0.40018455]
Mid axis: [0.98963519 0.11051238 0.09170172]
Short axis: [-0.12805234  0.39007625  0.91183503]
Scene points from depth: 0
Target points: 2087
Obstacle points used for scene clearance: 0


Number of valid grasp candidates: 471

Top candidates:

Candidate 0
score: 0.9289
width: 0.0272
normal_angle: 179.22
dist_to_centroid: 0.0575
self_clearance_score: 1.0
self_blocking_count: 0
scene_clearance_score: 1.0
scene_blocking_count: 0
pca_score: 0.9912
align_long: 0.1254
align_mid: 0.9912
align_short: 0.042
reasons: ['good antipodal normals', 'good normal-axis alignment', 'less ideal grasp width', 'farther from object centroid', 'reachable height region', 'good alignment with object minor axis', 'clear self approach', 'clear scene approach']

Candidate 1
score: 0.9042
width: 0.0274
normal_angle: 176.88
dist_to_centroid: 0.0573
self_clearance_score: 1.0
self_blocking_count: 0
scene_clearance_score: 1.0
scene_blocking_count: 0
pca_score: 0.9552
align_long: 0.0062
align_mid: 0.2959
align_short: 0.9552
reasons: ['good antipodal normals', 'good normal-axis alignment', 'less ideal grasp width', 'farther from object centroid', 'reachable height region', 'good alignment with object mino

In [130]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']